# Temporal ensemble optimisation

This notebook compares histogram-based gradient boosting (HGB) with a five-seed CatBoost ensemble and blends their predicted fraud probabilities. The aim is to find a combination that performs consistently across chronological validation periods before generating a Kaggle submission.

Only anonymised `feat_` columns are used. Models are fitted on transactions earlier than each validation window, and the final models are refitted on all labelled training rows before predicting `test.csv`. Graph features are not included.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'raw'
OUT = ROOT / 'submissions'
OUT.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
features = [c for c in train.columns if c.startswith('feat_')]
labelled = train.dropna(subset=['label']).copy()
assert 'time_step' in labelled.columns
assert labelled['time_step'].notna().all()
assert set(labelled['label'].unique()).issubset({0, 1})
assert all(c in test.columns for c in features)
print('Labelled:', labelled.shape, 'Test:', test.shape, 'Features:', len(features))
print('Time steps:', labelled.time_step.min(), labelled.time_step.max())

Labelled: (31235, 168) Test: (15329, 168) Features: 165
Time steps: 1 35


## Model configurations

HGB uses the same settings as the earlier gradient-boosting submission notebook. CatBoost is trained with five random seeds to examine whether averaging reduces sensitivity to random initialization. The CatBoost settings here were chosen for this comparison; they are not the result of a separate tuning search.

In [3]:
SEEDS = [11, 42, 73, 101, 202]
PERIODS = [(15, 19), (20, 24), (25, 29)]
CAT_PARAMS = dict(iterations=700, depth=6, learning_rate=0.045,
                  l2_leaf_reg=5, loss_function='Logloss', verbose=False,
                  thread_count=-1)

def fit_hgb(x, y):
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                      max_leaf_nodes=31, l2_regularization=1.0,
                                      random_state=42)
    m.fit(x, y)
    return m

def fit_cat(x, y, seed):
    m = CatBoostClassifier(**CAT_PARAMS, random_seed=seed)
    m.fit(x, y)
    return m

def score(y, p):
    return dict(pr_auc=average_precision_score(y, p),
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan)


## Chronological validation

The validation windows are steps 15–19, 20–24 and 25–29. For each window, training uses only labelled transactions from earlier steps. Both models are evaluated on the same rows using average precision (PR-AUC) and ROC-AUC.

CatBoost performs slightly better on steps 15–19 and 25–29, while HGB is stronger on steps 20–24. This variation suggests that neither model is consistently better across time.

In [4]:
predictions = {}
rows = []
for lo, hi in PERIODS:
    tr = labelled[labelled.time_step < lo]
    va = labelled[labelled.time_step.between(lo, hi)]
    if tr.empty or va.empty or tr.label.nunique() < 2 or va.label.nunique() < 2:
        print('Skipping unusable split:', (lo, hi), len(tr), len(va))
        continue
    xtr, ytr = tr[features], tr.label.astype(int)
    xva, yva = va[features], va.label.astype(int)
    baseline = fit_hgb(xtr, ytr).predict_proba(xva)[:, 1]
    cat_preds = np.column_stack([fit_cat(xtr, ytr, seed).predict_proba(xva)[:, 1]
                                 for seed in SEEDS])
    cat_mean = cat_preds.mean(axis=1)
    period = f'{lo}-{hi}'
    predictions[period] = dict(y=yva.to_numpy(), hgb=baseline,
                               cat_each=cat_preds, cat_mean=cat_mean,
                               n_train=len(tr), n_valid=len(va))
    for name, p in [('HGB baseline', baseline), ('CatBoost seed ensemble', cat_mean)]:
        rows.append(dict(period=period, model=name, n_train=len(tr),
                         n_valid=len(va), fraud_rate=yva.mean(), **score(yva, p)))
validation_results = pd.DataFrame(rows)
display(validation_results)


,period,model,n_train,n_valid,fraud_rate,pr_auc,roc_auc
0,15-19,HGB baseline,14896,3093,0.163595,0.987262,0.995038
1,15-19,CatBoost seed ensemble,14896,3093,0.163595,0.988448,0.994105
2,20-24,HGB baseline,17989,5617,0.126046,0.880880,0.952049
3,20-24,CatBoost seed ensemble,17989,5617,0.126046,0.871870,0.935112
4,25-29,HGB baseline,23606,2775,0.234955,0.991189,0.997044
5,25-29,CatBoost seed ensemble,23606,2775,0.234955,0.992611,0.997157


## Blend weights

The blend averages HGB and CatBoost probabilities using CatBoost weights of 0, 0.25, 0.50, 0.75 and 1. The selection criterion is mean validation PR-AUC across the three periods.

A CatBoost weight of **0.25** produces the highest mean PR-AUC (**0.953285**), narrowly ahead of HGB alone (**0.953110**). The blend improves steps 15–19 and 25–29 but performs slightly worse on steps 20–24. The difference in mean PR-AUC is small, so this should be interpreted as a modest validation improvement rather than a reliable gain across every period.

In [5]:
BLEND_WEIGHTS = [0.0, 0.25, 0.50, 0.75, 1.0]  # CatBoost weight
blend_rows = []
for period, d in predictions.items():
    for w in BLEND_WEIGHTS:
        p = w * d['cat_mean'] + (1 - w) * d['hgb']
        blend_rows.append(dict(period=period, cat_weight=w, n_valid=d['n_valid'],
                               **score(d['y'], p)))
blend_results = pd.DataFrame(blend_rows)
display(blend_results.pivot(index='cat_weight', columns='period', values='pr_auc'))
summary = (blend_results.groupby('cat_weight')
           .agg(mean_pr_auc=('pr_auc', 'mean'), min_pr_auc=('pr_auc', 'min'),
                mean_roc_auc=('roc_auc', 'mean')).reset_index())
display(summary.sort_values(['mean_pr_auc', 'min_pr_auc'], ascending=False))
# Predeclared selection rule: maximise mean period PR-AUC, break ties on worst period.
selected_weight = float(summary.sort_values(['mean_pr_auc', 'min_pr_auc'],
                                            ascending=False).iloc[0].cat_weight)
print('Selected CatBoost weight:', selected_weight)
print('Note: selection and reporting use the same validation periods; these are not unbiased final estimates.')

period,15-19,20-24,25-29
cat_weight,,,
0.00,0.987262,0.880880,0.991189
0.25,0.988270,0.878462,0.993123
0.50,0.988604,0.876565,0.993646
0.75,0.988692,0.874458,0.993411
1.00,0.988448,0.871870,0.992611


,cat_weight,mean_pr_auc,min_pr_auc,mean_roc_auc
1,0.25,0.953285,0.878462,0.979766
0,0.00,0.953110,0.880880,0.981377
2,0.50,0.952938,0.876565,0.978457
3,0.75,0.952187,0.874458,0.977091
4,1.00,0.950976,0.871870,0.975458


Selected CatBoost weight: 0.25
Note: selection and reporting use the same validation periods; these are not unbiased final estimates.


## Variation across CatBoost seeds

The five CatBoost runs show some variation in PR-AUC, particularly on steps 20–24. Averaging their predictions does not outperform every individual seed in every period, but provides a single prediction that is less dependent on choosing one random seed. These runs share the same training and validation data, so they are not independent tests of generalisation.

In [6]:
seed_rows = []
for period, d in predictions.items():
    for j, seed in enumerate(SEEDS):
        seed_rows.append(dict(period=period, seed=seed,
                              **score(d['y'], d['cat_each'][:, j])))
    seed_rows.append(dict(period=period, seed='ensemble',
                          **score(d['y'], d['cat_mean'])))
seed_results = pd.DataFrame(seed_rows)
display(seed_results.pivot(index='seed', columns='period', values='pr_auc'))

period,15-19,20-24,25-29
seed,,,
11,0.987590,0.867922,0.991886
42,0.989604,0.871037,0.991609
73,0.986955,0.872501,0.992962
101,0.988145,0.869497,0.992632
202,0.988416,0.873104,0.991804
ensemble,0.988448,0.871870,0.992611


## Final training and submission

The selected blend is refitted on all labelled rows in `train.csv`. HGB and the five CatBoost models predict the unlabelled transactions in `test.csv`; the final probabilities combine **75% HGB and 25% CatBoost**. The submission is saved with the required `index,target` columns.

This notebook generates the submission, which received a **0.94756 public leaderboard score**. The public score is a separate check, not the criterion used to select the blend weight.

In [7]:
X = labelled[features]
y = labelled.label.astype(int)
Xt = test[features]
hgb_final = fit_hgb(X, y)
hgb_test = hgb_final.predict_proba(Xt)[:, 1]
if selected_weight > 0:
    cat_test = np.column_stack([fit_cat(X, y, seed).predict_proba(Xt)[:, 1]
                                for seed in SEEDS]).mean(axis=1)
else:
    cat_test = np.zeros(len(test))
final_prob = (1 - selected_weight) * hgb_test + selected_weight * cat_test
assert sample.columns.tolist() == ['index', 'target']
assert len(sample) == len(final_prob)
assert np.isfinite(final_prob).all() and ((final_prob >= 0) & (final_prob <= 1)).all()
submission = sample.copy()
submission['target'] = final_prob
path = OUT / 'experiment12_temporal_ensemble.csv'
submission.to_csv(path, index=False)
print('Saved:', path, 'Shape:', submission.shape, 'CatBoost weight:', selected_weight)
display(submission.head())

Saved: /Users/charlenephua/Library/Mobile Documents/com~apple~CloudDocs/uni/Y4/y4s1/bt4012/bt4012-fraud-detection/submissions/experiment12_temporal_ensemble.csv Shape: (15329, 2) CatBoost weight: 0.25


,index,target
0,0,0.000454
1,1,0.000962
2,2,0.000883
3,3,0.000304
4,4,0.000272


## Findings

The 75:25 blend achieved the best mean PR-AUC among the tested weights, although its advantage over HGB alone was small and not uniform across validation periods. CatBoost seed averaging also showed that results depend somewhat on random initialization. These findings motivated the later use of a learned stacking model rather than relying on a fixed blend weight.

The validation windows were also used to select the weight, so their reported scores are model-selection results rather than independent estimates of final test performance.